In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")


Libraries loaded ✓


In [3]:
PATH = '../data/raw/'

orders    = pd.read_csv(PATH + 'olist_orders_dataset.csv')
items     = pd.read_csv(PATH + 'olist_order_items_dataset.csv')
payments  = pd.read_csv(PATH + 'olist_order_payments_dataset.csv')
customers = pd.read_csv(PATH + 'olist_customers_dataset.csv')
products  = pd.read_csv(PATH + 'olist_products_dataset.csv')
cat_trans = pd.read_csv(PATH + 'product_category_name_translation.csv')
reviews   = pd.read_csv(PATH + 'olist_order_reviews_dataset.csv')
sellers   = pd.read_csv(PATH + 'olist_sellers_dataset.csv')
geo       = pd.read_csv(PATH + 'olist_geolocation_dataset.csv')

for name, df in [('orders',orders),('items',items),('payments',payments),
                 ('customers',customers),('products',products)]:
    print(f"{name}: {df.shape}")

orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
customers: (99441, 5)
products: (32951, 9)


In [4]:
# Fix date columns
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date',
             'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Aggregate payments per order
pay_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value','sum'),
    payment_type=('payment_type','first')
).reset_index()

# Aggregate items per order
items_agg = items.groupby('order_id').agg(
    item_count=('order_item_id','count'),
    price=('price','sum'),
    freight_value=('freight_value','sum'),
    product_id=('product_id','first')
).reset_index()

# Aggregate reviews per order
rev_agg = reviews.groupby('order_id').agg(
    review_score=('review_score','mean')
).reset_index()

# Add English category names
products_en = products.merge(cat_trans, on='product_category_name', how='left')

# Master merge
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(pay_agg, on='order_id', how='left')
df = df.merge(items_agg, on='order_id', how='left')
df = df.merge(products_en[['product_id',
              'product_category_name_english']], on='product_id', how='left')
df = df.merge(rev_agg, on='order_id', how='left')

print(f"Master df shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Master df shape: (99441, 20)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_value', 'payment_type', 'item_count', 'price', 'freight_value', 'product_id', 'product_category_name_english', 'review_score']


In [5]:
# Keep only delivered orders
df = df[df['order_status'] == 'delivered'].copy()

# Derived columns
df['delivery_days'] = (
    df['order_delivered_customer_date'] -
    df['order_purchase_timestamp']).dt.days

df['estimated_days'] = (
    df['order_estimated_delivery_date'] -
    df['order_purchase_timestamp']).dt.days

df['late_delivery'] = df['delivery_days'] > df['estimated_days']
df['order_month']   = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_year']    = df['order_purchase_timestamp'].dt.year
df['order_quarter'] = df['order_purchase_timestamp'].dt.to_period('Q').astype(str)

# Remove rows with no payment value
df = df.dropna(subset=['payment_value'])

print(f"Final shape: {df.shape}")
print(f"Late deliveries: {df['late_delivery'].sum()} ({df['late_delivery'].mean()*100:.1f}%)")
print(f"Date range: {df['order_purchase_timestamp'].min().date()} to {df['order_purchase_timestamp'].max().date()}")

# Export
df.to_csv('../data/cleaned/olist_master.csv', index=False)
print("\n✓ Saved to data/cleaned/olist_master.csv")

Final shape: (96477, 26)
Late deliveries: 7306 (7.6%)
Date range: 2016-10-03 to 2018-08-29

✓ Saved to data/cleaned/olist_master.csv


In [6]:
# Reload and fix late_delivery column
df = pd.read_csv(r'C:\Users\ALG\OneDrive\Documents\olist-ecommerce-analysis\data\cleaned\olist_master.csv')

# Replace True/False with Late/On Time
df['delivery_status'] = df['late_delivery'].map({True: 'Late', False: 'On Time'})

# Save
df.to_csv(r'C:\Users\ALG\OneDrive\Documents\olist-ecommerce-analysis\data\cleaned\olist_master.csv', index=False)
print("✓ Done! delivery_status column added")
print(df['delivery_status'].value_counts())


✓ Done! delivery_status column added
delivery_status
On Time    89171
Late        7306
Name: count, dtype: int64
